# Lab 03 — Bag of Words (BoW)

**Curso:** Diplomado en Inteligencia Artificial  
**Nivel:** Introductorio–Intermedio  
**Duración estimada:** 45–60 minutos

---

## Contexto

**Bag of Words (BoW)** es la representación más simple de texto para modelos de machine learning.  
La idea: **ignorar el orden de las palabras y contar cuántas veces aparece cada una**.  
El resultado es un vector de conteos — un documento se convierte en un punto en un espacio de |V| dimensiones.

**Cuándo se usa:** clasificación de texto, detección de spam, análisis de sentimiento, búsqueda de documentos.

> ⚠️ **Limitación central:** `"no es bueno"` y `"bueno es no"` producen el **mismo vector BoW**.  
> El orden de las palabras desaparece por completo.

In [ ]:
# ⚙️ Configuración — Ejecuta esta celda primero
# Instala las dependencias necesarias para el lab
!pip install scikit-learn pandas --quiet

import re
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

print("✅ Librerías cargadas correctamente")

---

## Parte 1 — Cálculo Manual

Antes de usar ninguna librería, vamos a construir una matriz BoW **a mano** con un mini-corpus  
inspirado en el corpus de Wikipedia que usaremos en las siguientes partes.

### Mini-corpus (3 documentos)

| Doc | Texto |
|-----|-------|
| D1  | `andorra es un estado pequeño` |
| D2  | `andorra tiene capital en andorra` |
| D3  | `españa es un estado europeo` |

### Vocabulario

Todas las palabras únicas del corpus, ordenadas alfabéticamente:

`andorra · capital · en · es · estado · europeo · pequeño · tiene · un · españa`

---

### 📝 Ejercicio 1a — Construye la matriz a mano

Cuenta cuántas veces aparece cada palabra en cada documento y rellena las celdas `__`:

| Doc | andorra | capital | en | es | estado | europeo | pequeño | tiene | un | españa |
|-----|---------|---------|----|-----|--------|---------|---------|-------|----|--------|
| D1  | __      | __      | __ | __  | __     | __      | __      | __    | __ | __     |
| D2  | __      | __      | __ | __  | __     | __      | __      | __    | __ | __     |
| D3  | __      | __      | __ | __  | __     | __      | __      | __    | __ | __     |

> 💡 Tip: en D2 la palabra `"andorra"` aparece **dos veces** — ¿lo notaste?

In [ ]:
# Ejercicio 1a — Verifica tu respuesta con CountVectorizer
# Descomenta las líneas y ejecuta para comparar con tu tabla manual

# mini_corpus = [
#     "andorra es un estado pequeño",
#     "andorra tiene capital en andorra",
#     "españa es un estado europeo",
# ]
# v = CountVectorizer()
# X = v.fit_transform(mini_corpus)
# df = pd.DataFrame(X.toarray(),
#                   columns=v.get_feature_names_out(),
#                   index=["D1", "D2", "D3"])
# print(df)

---

### 📝 Ejercicio 1b — Preguntas de análisis

Responde en la siguiente celda de texto (doble clic para editar):

> **Pregunta 1:** Mirando solo los vectores, ¿qué par es más similar: (D1, D2) o (D1, D3)? ¿Por qué?

> **Pregunta 2:** D1 y D3 no mencionan los mismos países, pero comparten `"es"`, `"estado"` y `"un"`.  
> ¿Qué nos dice eso del tema de ambos documentos?

> **Pregunta 3:** Considera D4 = `"pequeño estado es andorra un"`.  
> ¿Cómo se vería su vector BoW comparado con D1? ¿Qué problema ves?

*(Escribe tus respuestas aquí)*

**Pregunta 1:** ___

**Pregunta 2:** ___

**Pregunta 3:** ___

---

## Parte 2 — Preprocesamiento del Corpus

El corpus que usaremos es un extracto de Wikipedia en español (`eswiki_corpus.txt`).  
Antes de aplicar BoW necesita **limpieza** porque contiene:

```
&lt;ref&gt;Polibio Historias III, 35&lt;/ref&gt;   ← referencias HTML escapadas
[[Iglesia de Sant Joan de Caselles]]         ← enlaces internos Wiki
''texto en cursiva'' y '''negrita'''         ← formato Wiki
&amp;nbsp; &lt;ref&gt; &quot;                ← entidades HTML
{{plantilla|parámetro=valor}}               ← plantillas wiki
```

> ⚠️ **Si estás en Google Colab:** ejecuta primero la celda siguiente para subir el archivo o montarlo desde Drive.  
> Si estás en local (VS Code / Jupyter), puedes saltar esa celda — el archivo se lee directamente.

In [ ]:

import os

# ---------------------------------------------------------------
# OPCIÓN A — Detectar entorno y cargar el archivo automáticamente
# ---------------------------------------------------------------
# Si estás en Colab, monta tu Google Drive y coloca eswiki_corpus.txt en
# la ruta indicada abajo. Luego descomenta el bloque correspondiente.
# Si estás en local (VS Code / Jupyter), no necesitas hacer nada.

EN_COLAB = 'google.colab' in str(get_ipython())

if EN_COLAB:
    # --- OPCIÓN A1: Montar Google Drive ---
    # Sube eswiki_corpus.txt a tu Drive (ej. en /Mi unidad/PLN/)
    # y ajusta la ruta CORPUS_FILE abajo.

    # from google.colab import drive
    # drive.mount('/content/drive')
    # CORPUS_FILE = '/content/drive/MyDrive/PLN/eswiki_corpus.txt'

    # --- OPCIÓN A2: Subir el archivo directamente ---
    from google.colab import files
    print("Sube el archivo eswiki_corpus.txt cuando aparezca el selector:")
    uploaded = files.upload()
    CORPUS_FILE = list(uploaded.keys())[0]

else:
    # Local: ruta relativa al repositorio
    CORPUS_FILE = os.path.join(os.path.dirname(os.path.abspath('__file__')),
                               '..', 'eswiki_corpus.txt')

print(f"✅ Ruta del corpus: {CORPUS_FILE}")
print(f"   Existe: {os.path.exists(CORPUS_FILE)}")


In [ ]:
# ⚙️ Función de limpieza + carga del corpus desde archivo
# Ejecuta esta celda antes de continuar

def limpiar(texto):
    """Elimina ruido típico de un dump de Wikipedia en español."""
    texto = re.sub(r'&lt;.*?&gt;', ' ', texto)           # tags HTML escapados
    texto = re.sub(r'<[^>]+>', ' ', texto)               # tags HTML reales
    texto = re.sub(r'\[\[.*?\]\]', ' ', texto)           # [[wiki links]]
    texto = re.sub(r'\{\{.*?\}\}', ' ', texto)           # {{plantillas}}
    texto = re.sub(r"'{2,}", '', texto)                  # ''cursiva'' '''negrita'''
    texto = re.sub(r'&\w+;', ' ', texto)                 # &amp; &nbsp; &quot;
    texto = re.sub(r'[^a-záéíóúüñA-ZÁÉÍÓÚÜÑ\s]', ' ', texto)  # solo letras
    texto = re.sub(r'\s+', ' ', texto).strip().lower()
    return texto


# Leer el corpus desde el archivo
N_LINEAS = 50   # Cambia este número para usar más o menos oraciones

with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
    lineas_raw = [l.strip() for l in f if l.strip()]

CORPUS_RAW = lineas_raw[:N_LINEAS]

# Limpiar
oraciones_limpias = [limpiar(o) for o in CORPUS_RAW]
oraciones_limpias = [o for o in oraciones_limpias if len(o.split()) > 3]

print(f"✅ Corpus cargado: {len(CORPUS_RAW)} líneas leídas → {len(oraciones_limpias)} oraciones válidas")
print(f"\nEjemplo (línea 3):")
print(f"  ANTES:   {CORPUS_RAW[2][:120]}")
print(f"  DESPUÉS: {oraciones_limpias[2][:120]}")


### 📝 Ejercicio 2 — Observa la función de limpieza

Descomenta el bloque siguiente y ejecuta para ver el antes/después de cada tipo de ruido.

In [ ]:
# Ejercicio 2 — Descomenta y ejecuta

# ejemplos = [
#     "Andorra &lt;ref&gt;ver nota&lt;/ref&gt; es un [[Estado]] ''soberano''.",
#     "Su población es de 85.101&amp;nbsp;habitantes (2024).",
#     "{{plantilla|tipo=país|nombre=Andorra}} fundado en 1278.",
#     "Limita con [[España]] y con [[Francia]] al norte.",
# ]

# for e in ejemplos:
#     print(f"  ANTES:   {e}")
#     print(f"  DESPUÉS: {limpiar(e)}")
#     print()

---

## Parte 3 — BoW con `CountVectorizer`

`CountVectorizer` de sklearn construye automáticamente el vocabulario y la matriz de conteos.  
Internamente hace lo mismo que hiciste a mano en la Parte 1, pero para miles de palabras.

### 📝 Ejercicio 3a — Unigramas sobre el corpus

Descomenta el bloque y ejecuta. Observa la forma de la matriz y las primeras filas.

In [ ]:
# Ejercicio 3a — Descomenta y ejecuta

# vectorizador = CountVectorizer(max_features=100)
# X = vectorizador.fit_transform(oraciones_limpias)
# df = pd.DataFrame(X.toarray(), columns=vectorizador.get_feature_names_out())

# print(f"Forma de la matriz: {X.shape}")
# print(f"  → {X.shape[0]} documentos  ×  {X.shape[1]} palabras del vocabulario")
# print()
# print(df.iloc[:5, :8])   # primeras 5 filas, primeras 8 columnas

# Responde:
# ¿Qué significa que la matriz tenga forma (N, 100)?
# ¿Qué porcentaje de celdas tienen valor 0? (= dispersión / sparsity)

---

### 📝 Ejercicio 3b — Top-10 palabras más frecuentes

Descomenta el bloque y observa qué palabras dominan el corpus.

In [ ]:
# Ejercicio 3b — Descomenta y ejecuta (requiere haber ejecutado 3a)

# conteo_total = X.toarray().sum(axis=0)
# palabras     = vectorizador.get_feature_names_out()
# top10        = sorted(zip(palabras, conteo_total), key=lambda x: -x[1])[:10]

# print("Top-10 palabras más frecuentes:")
# for palabra, freq in top10:
#     print(f"  {palabra:<20} {int(freq):>4} apariciones")

# Responde:
# ¿Son estas palabras útiles para entender el CONTENIDO de los documentos?
# ¿Las llamarías "palabras clave" de los artículos?

---

## Parte 4 — Stop Words

**Stop words** son palabras tan frecuentes en cualquier texto que no aportan información  
sobre el tema: *el, la, de, en, que, se, un, con, por…*

`CountVectorizer` puede ignorarlas automáticamente con el parámetro `stop_words`.

### 📝 Ejercicio 4 — Eliminar stop words y comparar

In [ ]:
# Lista de stop words en español
STOPWORDS_ES = [
    'a', 'al', 'algo', 'algunas', 'algunos', 'ante', 'antes', 'como',
    'con', 'contra', 'cual', 'cuando', 'de', 'del', 'desde', 'donde',
    'durante', 'e', 'el', 'ella', 'ellas', 'ellos', 'en', 'entre',
    'era', 'es', 'esa', 'esas', 'ese', 'eso', 'esos', 'esta', 'estas',
    'este', 'estos', 'fue', 'ha', 'hay', 'he', 'la', 'las', 'le', 'les',
    'lo', 'los', 'mas', 'me', 'mi', 'muy', 'ni', 'no', 'nos', 'o', 'os',
    'para', 'pero', 'por', 'que', 'se', 'si', 'sin', 'sobre', 'son',
    'su', 'sus', 'tambien', 'tanto', 'te', 'tiene', 'tu', 'tus',
    'un', 'una', 'uno', 'unos', 'unas', 'y', 'ya', 'yo', 'ser', 'han',
    'sido', 'parte', 'dicho', 'tras',
]

# Ejercicio 4 — Descomenta y ejecuta

# vectorizador_sw = CountVectorizer(max_features=100, stop_words=STOPWORDS_ES)
# X_sw            = vectorizador_sw.fit_transform(oraciones_limpias)

# conteo_sw   = X_sw.toarray().sum(axis=0)
# palabras_sw = vectorizador_sw.get_feature_names_out()
# top10_sw    = sorted(zip(palabras_sw, conteo_sw), key=lambda x: -x[1])[:10]

# print("Top-10 CON stop words eliminadas:")
# for i, (palabra, freq) in enumerate(top10_sw, 1):
#     print(f"  {i:2}. {palabra:<20} {int(freq):>4} apariciones")

# Responde:
# ¿Cuántas palabras del top-10 original desaparecen?
# ¿Las nuevas palabras son más informativas sobre el tema de los documentos?

---

## Parte 5 — N-gramas

Un **bigrama** es una secuencia de 2 palabras consecutivas.  
Captura cierta información de orden que el unigrama pierde:

| Tipo | Representación |
|------|----------------|
| Unigrama | `"andorra"`, `"la"`, `"vieja"` — tres tokens separados |
| Bigrama  | `"andorra la"`, `"la vieja"` — preserva la co-ocurrencia |

Con `ngram_range=(2, 2)` le decimos a `CountVectorizer` que use **solo bigramas**.  
Con `ngram_range=(1, 2)` usaría unigramas **y** bigramas.

### 📝 Ejercicio 5 — Bigramas sobre el corpus

In [ ]:
# Ejercicio 5 — Descomenta y ejecuta

# vectorizador_bg = CountVectorizer(ngram_range=(2, 2),
#                                   max_features=100,
#                                   stop_words=STOPWORDS_ES)
# X_bg            = vectorizador_bg.fit_transform(oraciones_limpias)

# conteo_bg   = X_bg.toarray().sum(axis=0)
# palabras_bg = vectorizador_bg.get_feature_names_out()
# top15_bg    = sorted(zip(palabras_bg, conteo_bg), key=lambda x: -x[1])[:15]

# print("Top-15 bigramas más frecuentes:")
# for i, (bigrama, freq) in enumerate(top15_bg, 1):
#     print(f"  {i:2}. {bigrama:<30} {int(freq):>4} apariciones")

# Responde:
# ¿Ves bigramas que corresponden a nombres propios o lugares?
# ¿Qué captura el bigrama "andorra vieja" que el unigrama no podría?

---

## 🤔 Reflexión Final

Responde en las celdas de texto a continuación (doble clic para editar).

---

**1. Orden ignorado**

Hemos visto que `D1 = "andorra es un estado pequeño"` y  
`D4 = "pequeño estado es andorra un"` producen el **mismo vector BoW**.  
¿Es esto siempre un problema, o hay casos donde no importa?

*(Escribe tu respuesta aquí)*

---

**2. Dispersión (sparsity)**

Con 25 documentos y vocabulario de 100 palabras, la mayoría de celdas son 0.  
En un corpus real con 50.000 documentos y vocabulario de 100.000 palabras:  
¿cuántos valores necesitarías almacenar sin matrices dispersas?  
¿Qué implica para la memoria?

*(Escribe tu respuesta aquí)*

---

**3. Relevancia ignorada**

BoW trata `"de"` (aparece 50 veces) y `"Pirineos"` (aparece 3 veces)  
de forma proporcional a sus conteos.  
¿Cuál de las dos es más informativa para identificar un documento sobre Andorra?  
¿Cómo podríamos penalizar las palabras muy frecuentes y premiar las específicas?

*(Escribe tu respuesta aquí)*

> La respuesta a la pregunta 3 es **TF-IDF** → continúa en el **Lab 04**.